# Week 6: MaskFormer tracking

Train your tracker, watch Hungarian matching at work, and measure tracking efficiency.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

In [ ]:
from exercises.week03.ex02_training_loop import LRConfig, train
from exercises.week06.ex02_build_tracker import build_tracker
from minihep.data.dataset import ToyTrackingDataset
from minihep.metrics import double_majority

data_dir = ensure_toy_data(num_train=200 if FAST else 2000, num_val=50 if FAST else 200)
fields = ["x", "y", "z", "r", "eta", "phi"]
cfg = {"inputs": {"hit": fields}, "targets": {"hit": ["on_valid_particle"], "particle": ["pt", "eta", "phi"]}, "event_max_num_particles": 32}
train_ds = ToyTrackingDataset(str(data_dir / "train"), **cfg)
val_ds = ToyTrackingDataset(str(data_dir / "val"), **cfg)
torch.manual_seed(0)
model = build_tracker(fields, dim=64, num_queries=32, num_encoder_layers=3, num_decoder_layers=3)
print(sum(p.numel() for p in model.parameters()), "parameters")

## The cost matrix before training

Rows are predictions (queries), columns targets. The matcher picks one cell per column (red dots).

In [ ]:
def show_matching(model, ds, i=0):
    model.eval()
    inputs, targets = ds[i]
    with torch.no_grad():
        out = model({k: v.to(DEVICE) for k, v in inputs.items()})
        t = {k: v.to(DEVICE) for k, v in targets.items()}
        final = out["final"]
        cost = sum(c for task in model.tasks for c in task.cost(final[task.name], t).values())[0].cpu()
        idx = model.matcher(cost[None], targets["particle_valid"])[0]
    n_true = int(targets["particle_valid"].sum())
    plt.imshow(cost[:, :n_true], aspect="auto", cmap="viridis")
    plt.colorbar(label="cost")
    plt.scatter(range(n_true), idx[:n_true], c="red", s=15)
    plt.xlabel("target particle")
    plt.ylabel("query")
    plt.show()
    model.train()


model.to(DEVICE)
show_matching(model, val_ds)

In [ ]:
steps = 300 if FAST else 6000
history = train(model, train_ds, steps, LRConfig(initial=1e-4, max=1e-3, end=1e-5, pct_start=0.05), device=DEVICE)
plt.plot(np.convolve(history.loss, np.ones(50) / 50, mode="valid"))
plt.yscale("log")
plt.title("training loss")
plt.show()
show_matching(model, val_ds)

## Efficiency, fakes and duplicates on the validation set

In [ ]:
import pandas as pd


@torch.no_grad()
def evaluate_tracking(model, ds):
    model.eval()
    rows, per_particle = [], []
    for i in range(len(ds)):
        inputs, targets = ds[i]
        out = model({k: v.to(DEVICE) for k, v in inputs.items()})
        preds = model.predict(out)["final"]
        pv, pm = preds["track_valid"]["track_valid"][0].cpu(), preds["track_hit_valid"]["track_hit_valid"][0].cpu()
        tv, tm = targets["particle_valid"][0], targets["particle_hit_valid"][0]
        rows.append(double_majority(pv, pm, tv, tm))
        # per-particle efficiency for an efficiency-vs-pT plot
        shared = pm[pv].long() @ tm.long().T
        n_track = pm[pv].sum(-1, keepdim=True).clamp_min(1)
        ok = ((shared / n_track > 0.5) & (shared / tm.sum(-1).clamp_min(1) > 0.5)).any(0)
        for p in range(int(tv.sum())):
            per_particle.append((targets["particle_pt"][0, p].item(), bool(ok[p])))
    model.train()
    return pd.DataFrame(rows).mean(), pd.DataFrame(per_particle, columns=["pt", "found"])


summary, per_particle = evaluate_tracking(model, val_ds)
print(summary)
bins = np.array([1, 1.5, 2, 3, 5, 10, 30])
per_particle["bin"] = pd.cut(per_particle.pt, bins)
eff = per_particle.groupby("bin", observed=False).found.mean()
plt.errorbar((bins[1:] + bins[:-1]) / 2, eff.values, xerr=(bins[1:] - bins[:-1]) / 2, fmt="o")
plt.xscale("log")
plt.ylim(0, 1.05)
plt.xlabel("particle pT [GeV]")
plt.ylabel("double-majority efficiency")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from minihep.plotting import event_display

inputs, targets = val_ds[0]
model.eval()
with torch.no_grad():
    preds = model.predict(model({k: v.to(DEVICE) for k, v in inputs.items()}))["final"]
pv, pm = preds["track_valid"]["track_valid"][0].cpu(), preds["track_hit_valid"]["track_hit_valid"][0].cpu()
track_label = np.zeros(pm.shape[1], dtype=int)
for q in torch.nonzero(pv).flatten():
    track_label[pm[q].numpy()] = int(q) + 1
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
truth_label = (targets["particle_hit_valid"][0].int().argmax(0) + 1) * targets["hit_on_valid_particle"][0]
event_display(inputs["hit_x"][0], inputs["hit_y"][0], inputs["hit_z"][0], truth_label.numpy(), "truth", axes[0])
event_display(inputs["hit_x"][0], inputs["hit_y"][0], inputs["hit_z"][0], track_label, "reconstructed", axes[1])
plt.show()
model.train()

## Your turn

1. Set `mask_attention=False` in the decoder (edit `build_tracker` or construct by hand) and retrain. Compare
   efficiency. This is the ablation that motivated Mask2Former.
2. Increase the hit count per event (`EventConfig(mean_num_particles=60)` in `minihep/utils.py` with a new
   data folder). Time the matcher with `%timeit` on the cost matrix. How does it scale?
3. Where in hepattn does `has_first_layer_loss: false` come in, and what would happen to layer 0's loss with
   learned queries if it were true?